# TLM Pretrain (TLM-XS / TLM-S / TLM-M)



# Cell 1 - Imports & Device

In [ ]:
!pip install -q orjson

import sys, os, shutil
from pathlib import Path

_SRC_DATASET          = "/kaggle/input/tlm-s-code"
_CURRICULUM_DATASET   = "/kaggle/input/curriclum"
_PRECOMPUTED_DATASET  = "/kaggle/input/curriculum-precomputed"

if not os.path.exists('/kaggle/working/src'):
    shutil.copytree(_SRC_DATASET, '/kaggle/working/src')
sys.path.insert(0, '/kaggle/working')

import dataclasses
import functools
import math
import random
import re
from typing import Dict, List
import numpy as np

from src.model_config import ModelConfig, count_parameters
from src.model_variants import TLM_S
from src.tokenizer import TLM_Tokenizer
from src.data_set import (
    iter_raw_code,
    make_sr_corruption,
    MAX_SRC_LEN,
    MAX_TGT_LEN,
    MAX_MASK_SPANS,
)
from src.data_loaders import (
    collate_fn,
    load_curriculum_config,
    tiers_for_epoch,
    max_epoch_from_curriculum,
    make_concat_dataset,
    curriculum_total_optimizer_steps,
    load_precomputed_tiers,
)
from src.training import (
    TrainingMetrics,
    ScheduledSampler,
    compute_loss,
    compute_loss_label_smoothed,
    compute_loss_free_pgen,
    _step_batch,
    run_epoch,
    validate,
    make_scheduler,
)

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import json

# --- Paths ---
_ON_KAGGLE = True

TOKENIZER_PATH    = "/kaggle/working/src/tokenizer/tokenizer_tlm_small.json"
_CURRICULUM_DIR   = Path(_CURRICULUM_DATASET)
TIER_PATHS        = [_CURRICULUM_DIR / f"tier{i}.jsonl" for i in range(1, 6)]
_PRECOMPUTED_ROOT = Path(_PRECOMPUTED_DATASET)
TIERS             = [1, 2, 3, 4, 5]
N_VARIANTS_TRAIN  = 5

# --- Reproducibility ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}  |  PyTorch: {torch.__version__}")
print("Running on: Kaggle")
print(f"Curriculum: {_CURRICULUM_DIR}")
for p in TIER_PATHS:
    status = "OK" if p.exists() else "MISSING"
    print(f"  {status}  {p.name}")
print(f"Precomputed corruption root: {_PRECOMPUTED_ROOT}")
for split in ("train", "val"):
    status = "OK" if (_PRECOMPUTED_ROOT / split).exists() else "MISSING"
    print(f"  {status}  {split}/")


# Cell 2 - Model Variant, Config, Model Def, Run Config

In [ ]:
# -- MODEL VARIANT -- set every run --
MODEL_VARIANT = "XS"   
assert MODEL_VARIANT in ("XXS", "XS", "S", "M"), (
    f"MODEL_VARIANT must be 'XXS', 'XS', 'S' or 'M', got {MODEL_VARIANT!r}"
)

if MODEL_VARIANT == "XXS":
    config = ModelConfig(d_model=96, d_ff=384, n_encoder_layers=3, n_decoder_layers=3,
                         n_heads=3, tier_name="TLM_XXS", target_param_count=1_740_000)
elif MODEL_VARIANT == "XS":
    config = ModelConfig(d_model=128, d_ff=512, tier_name="TLM_XS", target_param_count=3_130_000)
elif MODEL_VARIANT == "S":
    config = ModelConfig()
else:  # "M"
    config = ModelConfig(d_model=256, d_ff=1024, tier_name="TLM_M", target_param_count=9_500_000)

print(config)
print(f"Estimated params: {config.estimated_params()/1e6:.2f}M")
model = TLM_S(config)  # class is architecture-agnostic 

PAD_ID = config.special_token_ids["pad"]

# -- LABEL SMOOTHING -- 0.0 disables it (uses compute_loss_free_pgen, the unbiased copy gate loss)
LABEL_SMOOTHING = 0.0
if LABEL_SMOOTHING > 0:
    LOSS_FN = functools.partial(compute_loss_label_smoothed, smoothing=LABEL_SMOOTHING)
    print(f"Loss: compute_loss_label_smoothed (smoothing={LABEL_SMOOTHING})")
else:
    LOSS_FN = compute_loss_free_pgen
    print("Loss: compute_loss_free_pgen (unbiased copy gate, no smoothing)")

# -- RUN CONFIG -- set every run --
_base_run_name = f"tlm_{MODEL_VARIANT.lower()}_pretrain_unbiased_copy"
RUN_NAME = f"{_base_run_name}_ls" if LABEL_SMOOTHING > 0 else _base_run_name
RUN_MODE = "fresh"     # "fresh" (start new, random init) or "resume" (continue RUN_NAME's own last.pt)
assert RUN_MODE in ("fresh", "resume"), f"RUN_MODE must be 'fresh' or 'resume', got {RUN_MODE!r}"

CKPT_DIR = Path("/kaggle/working") / RUN_NAME
CKPT_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN_NAME={RUN_NAME!r}  RUN_MODE={RUN_MODE!r}")
print(f"  checkpoints to: {CKPT_DIR}")


## Cell 3 -- Tokenizer, Pre-tokenization & Tokenized Dataset

In [ ]:
# -- Load pre-trained tokenizer --
tokenizer = TLM_Tokenizer(vocab_size=config.vocab_size)
tokenizer.load(TOKENIZER_PATH)
print(f"Tokenizer loaded from {TOKENIZER_PATH}")
print(f"  vocab_size = {tokenizer.tokenizer.get_vocab_size()}")


## Cell 3b -- Smoke test

In [ ]:
# -- Smoke test --
_test_code = """\
def compute(alpha, beta, gamma):
    result = alpha * beta
    if result > gamma:
        return result - gamma
    return result + gamma
"""

_sentinel_ids = {config.special_token_ids[f"extra_id_{i}"] for i in range(MAX_MASK_SPANS)}
_SEARCH_ID    = config.special_token_ids["search"]
_REPLACE_ID   = config.special_token_ids["replace"]

_res = None
for _attempt in range(10):
    _res = make_sr_corruption(_test_code, tokenizer, config)
    if _res is not None:
        break

if _res is None:
    print("Smoke test: all 10 attempts returned None -- check choose_spans")
else:
    _enc, _dec = _res
    _n_sent   = sum(1 for t in _enc if t in _sentinel_ids)
    _n_search  = sum(1 for t in _dec if t == _SEARCH_ID)
    _n_replace = sum(1 for t in _dec if t == _REPLACE_ID)
    assert _n_sent == _n_search == _n_replace, (
        f"Mismatch: {_n_sent} sentinels in enc, {_n_search} SEARCH, {_n_replace} REPLACE in dec"
    )
    print(f"Smoke test passed (attempt {_attempt + 1})")
    print(f"  enc length : {len(_enc)}  (max {MAX_SRC_LEN})")
    print(f"  dec length : {len(_dec)}  (max {MAX_TGT_LEN})")
    print(f"  spans masked : {_n_sent}  |  SEARCH/REPLACE pairs : {_n_search}")
    print(f"  dec[:12] tokens : {[tokenizer.id_to_token(t) for t in _dec[:12]]}")


## Cell 4 -- Build per-tier datasets, DataLoaders & curriculum

In [ ]:
def load_codes_by_tier(split: str) -> Dict[int, List[str]]:
    """Collect raw code strings per tier for a split (only used for the qualitative
    mini-eval later -- training itself reads precomputed corruption pairs below)."""
    out: Dict[int, List[str]] = {}
    for p in TIER_PATHS:
        m = re.search(r"tier(\d+)", p.stem, flags=re.IGNORECASE)
        tier_no = int(m.group(1)) if m else len(out) + 1
        out[tier_no] = list(iter_raw_code(p, split_name=split))
    return out

VAL_CODES_BY_TIER = load_codes_by_tier("val")
val_codes = [c for t in sorted(VAL_CODES_BY_TIER) for c in VAL_CODES_BY_TIER[t]]
print(f"total val codes (raw, for qualitative eval): {len(val_codes):,}")

BATCH_SIZE  = 32
NUM_WORKERS = 2  
_PIN        = torch.cuda.is_available()
_collate    = lambda b: collate_fn(b, PAD_ID)

TRAIN_DS_BY_TIER = load_precomputed_tiers(_PRECOMPUTED_ROOT / "train", TIERS, N_VARIANTS_TRAIN)
VAL_DS_BY_TIER   = load_precomputed_tiers(_PRECOMPUTED_ROOT / "val",   TIERS, 1)

print("\nprecomputed examples per tier:")
for t in TIERS:
    print(f"  tier{t}: train {len(TRAIN_DS_BY_TIER[t]):,}  |  val {len(VAL_DS_BY_TIER[t]):,}")

val_ds = make_concat_dataset(VAL_DS_BY_TIER, TIERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=_collate, num_workers=NUM_WORKERS, pin_memory=_PIN)

CURRICULUM_CONFIG = load_curriculum_config(_CURRICULUM_DIR / "curriculum_config.json")
print("\nCurriculum schedule:")
for ep in range(1, max_epoch_from_curriculum(CURRICULUM_CONFIG) + 1):
    active = tiers_for_epoch(ep, CURRICULUM_CONFIG)
    n = sum(len(TRAIN_DS_BY_TIER[t]) for t in active if t in TRAIN_DS_BY_TIER)
    print(f"  epoch {ep:02d}: tiers {active} | train examples {n:,}")

_b = next(iter(DataLoader(
    make_concat_dataset(TRAIN_DS_BY_TIER, tiers_for_epoch(1, CURRICULUM_CONFIG)),
    batch_size=4, collate_fn=_collate)))
print("\nsample batch shapes:", {k: tuple(v.shape) for k, v in _b.items()})


In [ ]:
# -- Sanity overfit gate --
# 32 fixed examples (tier1, variant 0), dropout OFF.  PASS = nll < 0.3.
GATE_N     = 32
GATE_STEPS = 400
GATE_BS    = 8

gate_pairs = [TRAIN_DS_BY_TIER[1][i] for i in range(GATE_N)]
_gate_collate = lambda b: collate_fn(b, PAD_ID)
def _gate_loader():
    return DataLoader(gate_pairs, batch_size=GATE_BS, shuffle=True, collate_fn=_gate_collate)

_gate_cfg = dataclasses.replace(config, dropout=0.0)
gate_model = TLM_S(_gate_cfg).to(device)
gate_opt   = torch.optim.AdamW(gate_model.parameters(), lr=5e-4)

gate_model.train()
_last_nll = _last_pg = 0.0
for step in range(GATE_STEPS):
    s_nll = s_pg = n = 0
    for b in _gate_loader():
        loss, nll, pgen = _step_batch(gate_model, b, device, PAD_ID, loss_fn=LOSS_FN)
        gate_opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gate_model.parameters(), 1.0)
        gate_opt.step()
        s_nll += nll.item(); s_pg += pgen.item(); n += 1
    _last_nll, _last_pg = s_nll / n, s_pg / n
    if step % 25 == 0 or step == GATE_STEPS - 1:
        print(f"  step {step:3d} | nll {_last_nll:.4f} | "
              f"ppl {math.exp(min(_last_nll,20)):8.2f} | p_gen {_last_pg:.3f}")

print(f"\nGate result: nll={_last_nll:.4f}  p_gen={_last_pg:.3f}")
print("Overfit gate PASSED" if _last_nll < 0.3 else
      "Gate did NOT reach nll<0.3 -- inspect p_gen / capacity before full training")

del gate_model, gate_opt
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## Cell 8 -- Full curriculum pretraining loop (resumable, CSV/matplotlib metrics only)

In [ ]:

RESUME_PATH = CKPT_DIR / "last.pt"

EPOCHS       = max_epoch_from_curriculum(CURRICULUM_CONFIG)
LR           = 3e-4
WEIGHT_DECAY = 0.01
WARMUP_FRAC  = 0.03
PATIENCE     = 4
GRAD_CLIP    = 1.0
SS_ENABLED   = False   # hard rule: keep False (parallel SS breaks copy gate)
SS_DECAY     = "linear"

model = TLM_S(config).to(device)
print(f"Model params: {count_parameters(model)/1e6:.3f}M")

optimizer   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = curriculum_total_optimizer_steps(CURRICULUM_CONFIG, TRAIN_DS_BY_TIER, BATCH_SIZE)
warmup      = int(WARMUP_FRAC * total_steps)
scheduler   = make_scheduler(optimizer, warmup_steps=warmup, total_steps=total_steps)
sampler     = ScheduledSampler(total_epochs=EPOCHS, decay_type=SS_DECAY)

metrics = TrainingMetrics()
start_epoch = 1
best_val, bad_epochs = float("inf"), 0
best_ckpt = None

if RUN_MODE == "resume":
    assert RESUME_PATH.exists(), f"RUN_MODE='resume' but no last.pt at {RESUME_PATH}"
    print(f"Resuming from {RESUME_PATH}")
    _ckpt = torch.load(RESUME_PATH, map_location=device)
    model.load_state_dict(_ckpt["model"])
    optimizer.load_state_dict(_ckpt["optimizer"])
    scheduler.load_state_dict(_ckpt["scheduler"])
    start_epoch = _ckpt["epoch"] + 1
    best_val    = _ckpt["best_val"]
    bad_epochs  = _ckpt["bad_epochs"]
    best_ckpt   = _ckpt.get("best_ckpt")
    metrics.history = _ckpt["metrics_history"]
    print(f"  resuming at epoch {start_epoch}  |  best_val so far {best_val:.4f}  |  bad_epochs {bad_epochs}/{PATIENCE}")
else:
    assert not RESUME_PATH.exists(), (
        f"RUN_MODE='fresh' but {RESUME_PATH} already exists -- this would silently ignore "
        f"prior progress already saved under RUN_NAME={RUN_NAME!r}. Use RUN_MODE='resume' "
        f"to continue it, or pick a new RUN_NAME for a genuinely new run."
    )
    print("Starting fresh pretraining run")

print(f"total optimizer steps = {total_steps}  | warmup = {warmup}")
print(f"Scheduled sampling: {'ENABLED' if SS_ENABLED else 'disabled (teacher forcing)'}")

print("\n" + "=" * 70)
print(f"STARTING CURRICULUM PRETRAINING -- epoch {start_epoch}..{EPOCHS}")
print("=" * 70)

for epoch in range(start_epoch, EPOCHS + 1):
    active_tiers  = tiers_for_epoch(epoch, CURRICULUM_CONFIG)
    for t in active_tiers:
        TRAIN_DS_BY_TIER[t].set_variant(epoch - 1)   # cycle precomputed corruption variant per epoch
    train_dataset = make_concat_dataset(TRAIN_DS_BY_TIER, active_tiers)
    train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                               collate_fn=_collate, num_workers=NUM_WORKERS,
                               drop_last=True, pin_memory=_PIN,
                               persistent_workers=NUM_WORKERS > 0)

    print(f"\n=== Epoch {epoch:02d}/{EPOCHS} | tiers {active_tiers} | "
          f"train samples {len(train_dataset):,} | batches {len(train_loader)} ===")

    tr = run_epoch(model, train_loader, optimizer, scheduler, device, PAD_ID,
                   epoch=epoch, sampler=sampler, ss_enabled=SS_ENABLED,
                   grad_clip=GRAD_CLIP, log_every=200, loss_fn=LOSS_FN)
    va = validate(model, val_loader, device, PAD_ID, log_every=40, loss_fn=LOSS_FN)

    print(f"epoch {epoch:02d} | train loss {tr['loss']:.4f} | val loss {va['loss']:.4f} "
          f"| p_gen {tr['p_gen']:.3f}/{va['p_gen']:.3f} (train/val)")

    metrics.log(
        epoch=epoch, active_tiers=str(active_tiers),
        train_loss=tr["loss"], val_loss=va["loss"],
        train_pgen=tr["p_gen"], val_pgen=va["p_gen"], p_sample=tr["p_sample"],
    )

    if va["loss"] < best_val:
        best_val, bad_epochs = va["loss"], 0
        ckpt_path = CKPT_DIR / f"{RUN_NAME}_epoch_{epoch:02d}_val{va['loss']:.4f}.pt"
        torch.save({
            "model": model.state_dict(),
            "config": config.to_dict(),
            "epoch": epoch,
            "active_tiers": active_tiers,
            "curriculum_config": CURRICULUM_CONFIG,
            "ss_enabled": SS_ENABLED,
            "p_sample": tr["p_sample"],
            "val_loss": va["loss"],
            "val_pgen": va["p_gen"],
            "train_loss": tr["loss"],
            "metrics_history": dict(metrics.history),
        }, ckpt_path)
        best_ckpt = ckpt_path
        print(f"  saved {ckpt_path.name}  (best val loss so far)")
    else:
        bad_epochs += 1
        print(f"  no val improvement ({bad_epochs}/{PATIENCE})")

    # full resume state, overwritten every epoch regardless of improvement
    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "epoch": epoch,
        "best_val": best_val,
        "bad_epochs": bad_epochs,
        "best_ckpt": str(best_ckpt) if best_ckpt else None,
        "metrics_history": dict(metrics.history),
    }, RESUME_PATH)

    if bad_epochs >= PATIENCE:
        print("early stop")
        break

print("\n" + "=" * 70)
print("CURRICULUM PRETRAINING COMPLETE")
print(f"Best val loss : {best_val:.4f}")
print(f"Best ckpt     : {best_ckpt}")
print("=" * 70)


In [ ]:
import csv
import matplotlib.pyplot as plt

# -- 1. Save to CSV --
csv_path = CKPT_DIR / "training_curves.csv"
epochs = metrics.history["epoch"]
fieldnames = list(metrics.history.keys())

with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for i in range(len(epochs)):
        writer.writerow({k: metrics.history[k][i] for k in fieldnames})

print(f"Saved training curves to {csv_path}")

# -- 2. Plot loss --
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, metrics.history["train_loss"], label="train loss", marker="o")
ax1.plot(epochs, metrics.history["val_loss"],   label="val loss",   marker="o")
ax1.set_xlabel("epoch"); ax1.set_ylabel("loss (NLL + p_gen + coverage penalty)")
ax1.set_title("Loss curves"); ax1.legend(); ax1.grid(True)

# -- 3. Plot p_gen --
ax2.plot(epochs, metrics.history["train_pgen"], label="train p_gen", marker="o")
ax2.plot(epochs, metrics.history["val_pgen"],   label="val p_gen",   marker="o")
ax2.axhline(0.10, color="red",    linestyle="--", linewidth=0.8, label="collapse floor (0.10)")
ax2.axhline(0.45, color="green",  linestyle="--", linewidth=0.8, label="target (0.45)")
ax2.set_xlabel("epoch"); ax2.set_ylabel("p_gen")
ax2.set_title("Copy gate health"); ax2.legend(); ax2.grid(True)

plt.tight_layout()
fig_path = CKPT_DIR / "training_curves.png"
plt.savefig(fig_path, dpi=150)
plt.show()
print(f"Saved plot to {fig_path}")
